# Debug Player Move Scenario
This notebook simulates the issue "When moving a player from one session to another, nothing happens" by mocking simple Player and Session objects.
We will demonstrate the potential logic flaw where a player may not be properly transferred due to silent failures or state references.

In [ ]:
# 1. Define Player and Session Classes 
class Player:
    def __init__(self, name, session_id):
        self.name = name
        self.session_id = session_id

class Session:
    def __init__(self, session_id, players=None):
        self.session_id = session_id
        self.players = players if players is not None else []
    
    def add_player(self, player):
        if player not in self.players:
            self.players.append(player)
    
    def remove_player(self, player):
        if player in self.players:
            self.players.remove(player)

In [ ]:
# 2. Initialize Scenario
session_a = Session('session_a')
session_b = Session('session_b')

player_1 = Player('John Doe', 'session_a')
session_a.add_player(player_1)

print(f"Initial: Player {player_1.name} assigned to {player_1.session_id}")
print(f"Players in Session A: {[p.name for p in session_a.players]}")
print(f"Players in Session B: {[p.name for p in session_b.players]}")

In [ ]:
# 3. Simulate Buggy Transfer (Does not update session correctly)
def buggy_transfer(player, source, target):
    # This just updates the player object but fails to remove from source or add to target
    player.session_id = target.session_id
    print(f"Bug simulation: Updated player session ID to {player.session_id}, but forgot lists.")

buggy_transfer(player_1, session_a, session_b)

print(f"After buggy move: Player assigned to {player_1.session_id}")
print(f"Players in Session A (Should be empty): {[p.name for p in session_a.players]}")
print(f"Players in Session B (Should have player): {[p.name for p in session_b.players]}")


In [ ]:
# 4. Implement Robust Transfer Function
def robust_transfer(player, source, target):
    if player not in source.players:
        print(f"Error: Player not in source session {source.session_id}")
        return False
    
    # 1. Update Player Object
    player.session_id = target.session_id
    
    # 2. Add to New Session
    target.add_player(player)

    # 3. Remove from Old Session
    source.remove_player(player)

    print(f"Successfully transferred {player.name} from {source.session_id} to {target.session_id}")
    return True

# Reset Scenario first
source = session_a  # Current state: still in a list
target = session_b
# Resetting correct state for demo:
player_1.session_id = 'session_a' 
# Since buggy transfer didn't change lists, player is already in source list and not in target list


In [ ]:
# 5. Verify Successful Transfer
robust_transfer(player_1, session_a, session_b)

print(f"Final State:")
print(f"Player {player_1.name} assigned to {player_1.session_id}")
print(f"Players in Session A: {[p.name for p in session_a.players]}")
print(f"Players in Session B: {[p.name for p in session_b.players]}")

# Validate assertions
assert player_1.session_id == 'session_b', "Player ID not updated"
assert len(session_a.players) == 0, "Player not removed from Session A"
assert len(session_b.players) == 1, "Player not added to Session B"
print("All verifications passed.")